# Working with the KITTI Dataset

KITTI is a widely used robotic vision dataset recorded from a vehicle. It contains synchronized camera, GPS, IMU, and lidar measurements.

See the [KITTI website](https://www.cvlibs.net/datasets/kitti/) and [dataset paper](https://www.cvlibs.net/publications/Geiger2013IJRR.pdf) for more information.

This notebook introduces the data interface used in the ENN583 practicals and visual odometry assessment.

In [ ]:
# This code cell is responsible for importing the `kitti_utils` module, which provides utilities for working with the KITTI dataset.
# It first attempts to find the repository root by looking for the presence of the `kitti_utils.py` file in the current directory and its parent directories. If it finds the file, it adds the `support` directory to the Python path so that the module can be imported. 
# If it cannot find the file, it raises a RuntimeError.
from pathlib import Path
import sys

support_dir = (Path.cwd().resolve().parents[1] / "support")
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))
    
import kitti_utils as kitti

## Load a sequence

`load_kitti_dataset` downloads the selected drive and its calibration files when they are not already available. Data is stored in `data/kitti` by default.

In [ ]:
sequence = "2011_09_26_drive_0035"
dataset = kitti.load_kitti_dataset(sequence)

print("Frames:", dataset.frame_count)
print("Dataset type:", type(dataset).__name__)

## Stereo images

`stereo(i)` returns the synchronized left and right RGB images from the colour stereo cameras (cameras 2 and 3 in the dataset) as NumPy arrays.

In [ ]:
import matplotlib.pyplot as plt

sample_frames = range(0, dataset.frame_count, 20)
for frame in sample_frames:
    left, right = dataset.stereo(frame)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(left)
    axes[0].set_title(f"Camera 2, frame {frame}")
    axes[1].imshow(right)
    axes[1].set_title(f"Camera 3, frame {frame}")
    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
    plt.show()

## Camera calibration

The calibration dictionary contains the intrinsic matrix `K`, rectified projection matrix `P`, and transform `T_cam_imu` from the IMU frame to the selected camera frame.

We will learn about camera calibration and what these matrices mean in a later week.

In [ ]:
calibration = dataset.camera_calibration(camera=2)

print("K:\n", calibration["K"])
print("P:\n", calibration["P"])
print("T_cam_imu:\n", calibration["T_cam_imu"])

## Ground-truth trajectory

The full Kitti dataset also has the ground-truth poses for all frames. As you develop your solution to the coding assignment, you can use the ground truth for analysis and comparison. Each pose is a `spatialmath.SE3` transform relative to frame 0.

Ground truth is **not available** in the Gradescope environment when your code will be autograded. There, your methods will only see a restricted dataset object without the ground truth information.

In [ ]:
import numpy as np

poses = [dataset.ground_truth_pose(frame) for frame in range(dataset.frame_count)]
positions = np.array([pose.t for pose in poses])

plt.figure(figsize=(7, 5))
plt.plot(positions[:, 0], positions[:, 1])
plt.scatter(positions[0, 0], positions[0, 1], label="Start")
plt.title("Ground-truth trajectory")
plt.xlabel("X (m)"); plt.ylabel("Y (m)")
plt.axis("equal"); plt.grid(); plt.legend(); 
plt.show()

print("First pose:\n", poses[0])
print("Last pose:\n", poses[-1])

## Assessment interface

When your code runs in the Autograder environment, it will only have access to a restricted view of the dataset. 
The restricted view provides stereo images, calibration, and sequence length, but no ground-truth pose method.

In [ ]:
assessment_dataset = dataset.student_view()

print("Assessment dataset type:", type(assessment_dataset).__name__)
print("Frames:", len(assessment_dataset))
print("Stereo shape:", assessment_dataset.stereo(0)[0].shape)
print("Has ground_truth_pose:", hasattr(assessment_dataset, "ground_truth_pose"))
print("Has underlying data object:", hasattr(assessment_dataset, "data"))